# Disease Prediction from Symptoms

Map a set of symptoms to a likely disease. Synthetic dataset modeled on the Kaggle "Disease Prediction Using Machine Learning" structure: 17 symptom slots per row (each cell either a symptom name or blank), single disease label.

Despite the file name I'm doing this as **multi-class** (one label per row), not multi-label. The Kaggle dataset is also single-label; the "multilabel" in this notebook's folder name is mostly aspirational — if I had time I'd extend it to overlapping symptom sets where a patient could have two co-occurring conditions, but for now it's a multinomial classifier.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

np.random.seed(17)
df = pd.read_csv("../../data/disease_symptoms/disease_symptoms.csv")
print(df.shape)
df.head()


(4000, 18)


,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17,Disease
0,NaN,NaN,NaN,lethargy,NaN,bloody_stool,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,abnormal_menstruation,unsteadiness,NaN,Hepatitis B
1,NaN,weakness_in_limbs,NaN,NaN,cough,NaN,NaN,irregular_sugar_level,patches_in_throat,NaN,spinning_movements,NaN,continuous_sneezing,NaN,constipation,NaN,NaN,Diabetes
2,itching,NaN,dischromic_patches,NaN,NaN,NaN,foul_smell_of_urine,swelling_joints,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chicken pox
3,shivering,altered_sensorium,skin_rash,NaN,bloody_stool,NaN,dehydration,swelling_joints,NaN,NaN,pain_in_anal_region,NaN,NaN,NaN,NaN,NaN,NaN,Tuberculosis
4,NaN,NaN,NaN,NaN,NaN,weakness_in_limbs,NaN,irregular_sugar_level,spinning_movements,patches_in_throat,NaN,NaN,continuous_sneezing,blurred_and_distorted_vision,NaN,constipation,cough,Diabetes


In [2]:
# turn each row's symptom slots into a single space-separated string,
# then use CountVectorizer to get a multi-hot feature matrix
def row_to_text(row):
    return " ".join(s for s in row if isinstance(s, str) and s.strip())

symptom_cols = [c for c in df.columns if c.startswith("Symptom_")]
texts = df[symptom_cols].fillna("").apply(row_to_text, axis=1).values
y = df["Disease"].values

X_train, X_test, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, stratify=y, random_state=17
)

vec = CountVectorizer(token_pattern=r"[A-Za-z_()]+", binary=True, min_df=1)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)
print(f"vocab: {len(vec.vocabulary_)} symptoms, classes: {len(set(y))}")


vocab: 105 symptoms, classes: 30


## Two classifiers

In [3]:
clf_lr = LogisticRegression(max_iter=2000, C=2.0, n_jobs=-1)
clf_lr.fit(Xtr, y_train)
pred_lr = clf_lr.predict(Xte)

clf_rf = RandomForestClassifier(n_estimators=300, random_state=17, n_jobs=-1)
clf_rf.fit(Xtr, y_train)
pred_rf = clf_rf.predict(Xte)

print(f"LogReg accuracy:       {accuracy_score(y_test, pred_lr):.3f}")
print(f"RandomForest accuracy: {accuracy_score(y_test, pred_rf):.3f}")


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


LogReg accuracy:       1.000
RandomForest accuracy: 1.000


In [4]:
# spot-check: feed a custom symptom list and see what comes back
sample = "high_fever vomiting headache chills"
def predict(text, model=clf_lr, top_k=3):
    proba = model.predict_proba(vec.transform([text]))[0]
    top = np.argsort(proba)[-top_k:][::-1]
    return [(model.classes_[i], float(proba[i])) for i in top]

print(f"symptoms: {sample}")
for disease, p in predict(sample, top_k=5):
    print(f"  {disease:30s} {p:.3f}")


symptoms: high_fever vomiting headache chills
  Dengue                         0.080
  Hepatitis D                    0.072
  Hepatitis A                    0.068
  Allergy                        0.065
  Chicken pox                    0.064


## Reality check

A few caveats:

- This data is synthetic with one disease per row and a tight symptom-to-disease mapping. Real medical data has comorbidities, missing symptoms, and overlapping presentations. The numbers above are wildly optimistic.
- I'm using multinomial classification, not multi-label. Adding a sigmoid-per-disease head would make this truly multi-label, which would also let it say "no condition matches strongly" instead of forcing a choice.
- **Do not treat this as a diagnostic tool.** This is an exercise in mapping discrete symptom sets to categorical outputs. A real triage system needs probability calibration, uncertainty estimates, and most importantly a human in the loop.
